# 04. Centros de distribuição: rerandomização com poucas unidades

**Setor:** logística. **Decisão:** um novo roteirizador melhora a taxa de
entregas no prazo? As unidades são **apenas 24 centros de distribuição (CDs)**, e
eles diferem em volume (`throughput_pre`) e número de docas (`dock_count`). Com
tão poucas unidades, um único sorteio pode sair **desbalanceado por azar**,
concentrando os CDs grandes num braço.

A solução é **rerandomizar**: sortear repetidamente até que as covariáveis
fiquem equilibradas (critério de Mahalanobis), e depois usar uma **inferência de
randomização que respeita esse mesmo critério**. Base teórica:
[II. Desenhos](../../../docs/pt-br/teoria/02-desenhos.md) (tópico 6) e
[IV. Inferência](../../../docs/pt-br/teoria/04-inferencia.md) (tópico 13).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "logistics_dc.csv")
print(df.shape, "distribution centers")
df.head()


## 1. Um CRD simples pode desbalancear

Primeiro, um sorteio de `CRD` comum e o equilíbrio das covariáveis. Com 24
unidades, não é raro a SMD de `throughput_pre` ou `dock_count` passar de `0,1`.


In [ ]:
from skxperiments.design.crd import CRD
from skxperiments.design.balance import check_balance

crd = CRD(p=0.5, seed=404).randomize(df[["region", "throughput_pre", "dock_count"]].copy())
check_balance(crd)[["covariate", "smd"]].round(3)


## 2. Rerandomizar até equilibrar

`ReRandomizedCRD` aceita só os sorteios cuja distância de Mahalanobis fica abaixo
de um limiar. Usamos `chi2.ppf(0,10, df=2)` (aceita os ~10% sorteios mais
equilibrados; com `n` pequeno, um limiar não muito rígido evita rejeição
excessiva). O balanço depois deve estar visivelmente melhor.


In [ ]:
from scipy.stats import chi2

from skxperiments.design.rerandomized_crd import ReRandomizedCRD
from skxperiments.diagnostics import BalanceReport

threshold = float(chi2.ppf(0.10, df=2))
design = ReRandomizedCRD(
    covariates=["throughput_pre", "dock_count"], threshold=threshold,
    p=0.5, seed=404, max_attempts=20000,
)
assignment = design.randomize(df[["region", "throughput_pre", "dock_count"]].copy())
report = BalanceReport().run(assignment)
print("imbalanced covariates:", report.imbalanced)


In [ ]:
from skxperiments.reporting import plot_balance

ax = plot_balance(report)
ax.set_title("Balance after re-randomization")
ax.figure


## 3. Inferência que respeita o critério

Anexamos o resultado (taxa no prazo) e analisamos. O `RandomizationTest` re-sorteia
pelo **mesmo** mecanismo do desenho (via `draw()`), então as permutações também
obedecem ao critério de rerandomização, mantendo o teste válido (não
conservador). Com 24 unidades, `C(24,12)` é enorme, então o p-valor vem por Monte
Carlo. Comparamos com o intervalo de Neyman.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import RandomizationTest, NeymanCI

t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["on_time"] = np.where(t == 1, df["on_time_y1"].to_numpy(), df["on_time_y0"].to_numpy())
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=404
)

rand = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="on_time"), n_permutations=2000, seed=0
).fit(assignment).estimate()
neyman = NeymanCI(estimator=DifferenceInMeans(outcome_col="on_time")).fit(assignment).estimate()

print(f"Randomization test: ATE={rand.ate:.3f} pp, p={rand.p_value:.4f}")
print(f"Neyman CI:          ATE={neyman.ate:.3f} pp, "
      f"CI=({neyman.ci[0]:.3f}, {neyman.ci[1]:.3f})")


In [ ]:
from skxperiments.reporting import plot_null_distribution

ax = plot_null_distribution(rand)
ax.set_title("Randomization null distribution")
ax.figure


## Decisão

O efeito verdadeiro é `+1,5` ponto percentual na taxa no prazo. A estimativa fica
perto disso e o teste de randomização rejeita a nula. A lição de desenho: com
poucas unidades grandes, a rerandomização protege contra um sorteio azarado, e a
inferência **precisa** honrar o critério para não ficar conservadora. Um teste t
comum aqui seria válido mas menos poderoso.

Próximo passo: [05. Muitas métricas no streaming](05_streaming_many_metrics.ipynb).
